# Iterators and generators

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-04-iterators-and-generators.ipynb)

Data engineering is mostly about data that does not fit in memory. A list of ten million rows needs gigabytes; a
generator that yields one row at a time needs almost nothing. This chapter explains how iteration works, how to write
generators, and how the standard library `itertools` module composes them into pipelines.

## Iterables and iterators

An **iterable** is anything you can loop over: a list, a file, a dict, a `range`. Calling `iter()` on it gives an
**iterator**, an object whose `next()` returns the following item and raises `StopIteration` when it is finished.

`for` does this for you. Most iterators can be consumed only once.

### Where you'll use it

- **Reading files and database cursors:** a file object, a DB-API cursor and a Kafka consumer are all iterators. A `for` loop over them reads one record at a time, so memory stays flat.
- **Feeding APIs:** a paginated REST API is an iterator over pages. The loop asks for the next page only when it needs it.
- **Why it matters:** code written for a list also works on an iterator, so the same transform runs on a 1 KB test file and a 50 GB production file.

In [ ]:
numbers = [10, 20, 30]
it = iter(numbers)
assert next(it) == 10
assert next(it) == 20
assert next(it) == 30

try:
    next(it)
except StopIteration:
    pass
else:
    raise AssertionError("expected StopIteration")

# A list can be iterated many times; an iterator only once.
assert sum(numbers) == 60 and sum(numbers) == 60
single_use = iter(numbers)
assert sum(single_use) == 60 and sum(single_use) == 0

## Generators: `yield`

A function that contains `yield` is a **generator function**. Calling it returns a generator that produces values
on demand. Execution pauses at each `yield` and resumes on the next `next()`. Nothing runs until you ask for a value.

### Where you'll use it

- **Streaming ETL:** a generator reads, parses and yields one cleaned record at a time into a bulk writer.
- **Paginated APIs:** `yield from fetch_pages()` gives the caller every item across all pages as one stream.
- **Live feeds:** a generator that tails a log file and yields new lines as they are appended.

In [ ]:
def read_batches(total, size):
    start = 0
    while start < total:
        yield list(range(start, min(start + size, total)))
        start += size

batches = read_batches(7, 3)
assert next(batches) == [0, 1, 2]
assert list(batches) == [[3, 4, 5], [6]]

## Generator expressions

A generator expression looks like a list comprehension with parentheses. It computes items lazily. Use it when you
only need to loop once or pass the result straight to a function such as `sum`. Use a list comprehension when you need
to index or reuse the result.

### Where you'll use it

- **Aggregating large inputs:** `sum(x for x in rows if x.is_valid)` computes a total without building a list.
- **Passing to writers:** a generator expression feeding `csv.writer.writerows` writes millions of rows with constant memory.

In [ ]:
amounts = range(1, 1_000_001)

total = sum(a for a in amounts if a % 2 == 0)     # nothing is stored in memory
assert total == 250_000_500_000

squares = [a * a for a in range(5)]               # a list: indexable and reusable
assert squares[-1] == 16

## A streaming pipeline

Chain generators so each stage handles one record at a time. This is the in-memory version of the shape of a data
pipeline: read, parse, filter, transform. The whole input never sits in memory at once.

### Where you'll use it

- **Multi-stage transforms:** this chain (parse, validate, enrich) is the shape of most batch ETL jobs. Each stage can be tested on its own.
- **Back-pressure:** because each stage pulls from the previous one, a slow writer slows the reader instead of filling memory.

In [ ]:
raw_lines = [
    "ana,30,ok",
    "ben,x,ok",
    "cy,41,ok",
    "dee,25,error",
]

def parse(lines):
    for line in lines:
        name, age, status = line.split(",")
        yield {"name": name, "age": age, "status": status}

def valid_age(records):
    for record in records:
        if record["age"].isdigit():
            record["age"] = int(record["age"])
            yield record

def only_ok(records):
    return (r for r in records if r["status"] == "ok")

clean = list(only_ok(valid_age(parse(raw_lines))))
assert [r["name"] for r in clean] == ["ana", "cy"]
assert clean[1]["age"] == 41

## `itertools`: standard building blocks

`itertools` provides fast, memory-light tools that return iterators:

- `islice(it, n)` takes the first `n` items, even from an endless generator.
- `chain(a, b)` joins iterables without copying them.
- `groupby(items, key)` groups **consecutive** items that share a key. Sort by the key first if you want full groups.
- `accumulate` computes running totals.

### Where you'll use it

- **Taking a sample:** `islice` reads the first N rows of a huge file to inspect the schema.
- **Sessionising or grouping sorted events:** `groupby` on a user ID after sorting by `(user_id, ts)`.
- **Joining streams:** `chain` concatenates the output of several daily files into one iterator.

In [ ]:
from itertools import accumulate, chain, groupby, islice, count

first_five_odd = list(islice((n for n in count(1) if n % 2), 5))
assert first_five_odd == [1, 3, 5, 7, 9]

assert list(chain([1, 2], [3], [])) == [1, 2, 3]
assert list(accumulate([1, 2, 3, 4])) == [1, 3, 6, 10]

events = [("ana", 1), ("ana", 2), ("ben", 5), ("ana", 3)]
events.sort(key=lambda e: e[0])            # groupby needs sorted input
groups = {name: [v for _, v in items] for name, items in groupby(events, key=lambda e: e[0])}
assert groups == {"ana": [1, 2, 3], "ben": [5]}

## Chunking an iterator

Batching a stream into fixed-size chunks is one of the most common pipeline steps: it bounds memory and gives the
database writes a sensible size. `itertools.batched` does this from Python 3.12. This version works on any Python.

### Where you'll use it

- **Bulk database writes:** `executemany` or a COPY statement is much faster per batch of 1,000 rows than per row.
- **API limits:** sending 100 records per request to stay under a provider's payload limit.
- **Parallel work units:** splitting a stream into chunks that a worker pool processes concurrently.

In [ ]:
def chunked(iterable, size):
    if size < 1:
        raise ValueError("size must be at least 1")
    iterator = iter(iterable)
    while True:
        chunk = list(islice(iterator, size))
        if not chunk:
            return
        yield chunk

assert list(chunked(range(5), 2)) == [[0, 1], [2, 3], [4]]
assert list(chunked([], 3)) == []

try:
    list(chunked([1], 0))
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

## Practice questions

1. Write `running_max(values)` as a generator that yields the largest value seen so far.
2. Write `count_lines_longer_than(lines, limit)` that uses a generator expression and returns a count.
3. Use `groupby` to compress a list of states into `(state, run_length)` pairs, for example `["up","up","down"]` gives
   `[("up", 2), ("down", 1)]`.

## Answers

### Question 1

In [ ]:
def running_max(values):
    best = None
    for value in values:
        best = value if best is None or value > best else best
        yield best

assert list(running_max([3, 1, 4, 1, 5])) == [3, 3, 4, 4, 5]

### Question 2

In [ ]:
def count_lines_longer_than(lines, limit):
    return sum(1 for line in lines if len(line) > limit)

assert count_lines_longer_than(["a", "abcd", "abc"], 2) == 2

### Question 3

In [ ]:
from itertools import groupby

def run_lengths(states):
    return [(state, sum(1 for _ in run)) for state, run in groupby(states)]

assert run_lengths(["up", "up", "down"]) == [("up", 2), ("down", 1)]
assert run_lengths(["up", "down", "up"]) == [("up", 1), ("down", 1), ("up", 1)]

## Interview notes

- **Generators are one-pass.** A second loop over the same generator sees nothing. Store the result or re-create the
  generator when you need two passes.
- **Lazy means nothing runs until consumed.** A bug in a generator stage appears only when the final consumer pulls.
- **`groupby` groups consecutive items only.** Sorting first gives full groups. When the data does not fit in memory,
  sorting needs an external sort, and grouping belongs in a database `GROUP BY` or an engine such as Spark.
- **Summary.** Read and transform with generators, and hold only one batch at a time.